In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import mlflow.pytorch
from PIL import Image

from cancer_detection.data.metadata import MetadataEncoder
from cancer_detection.data.transforms import get_val_transforms
from cancer_detection.explainability.gradcam import GradCAMWrapper

%matplotlib inline

MODEL_URI = 'models:/melanoma-classifier/Production'
IMG_DIR = Path('../data/raw/train_images')
TEST_CSV = Path('../data/processed/test.csv')

In [ ]:
# Load model from MLflow registry
model = mlflow.pytorch.load_model(MODEL_URI)
model.eval()
grad_cam = GradCAMWrapper(model)
encoder = MetadataEncoder()
val_transform = get_val_transforms(384)
print('Model and GradCAM wrapper loaded')

In [ ]:
test_df = pd.read_csv(TEST_CSV)

def visualise_row(row, ax_orig, ax_cam, label_str):
    img_path = IMG_DIR / f"{row['image_name']}.jpg"
    original = np.array(Image.open(img_path).convert('RGB'))
    
    image_tensor = val_transform(image=original)['image']
    meta_tensor = encoder.encode(row)
    
    heatmap = grad_cam.generate_heatmap(image_tensor, meta_tensor, original, image_size=384)
    
    with torch.no_grad():
        logit = model(image_tensor.unsqueeze(0), meta_tensor.unsqueeze(0))
        prob = torch.sigmoid(logit).item()
    
    ax_orig.imshow(Image.fromarray(original).resize((256, 256)))
    ax_orig.set_title(f'{label_str}\nP(malignant)={prob:.3f}', fontsize=9)
    ax_orig.axis('off')
    
    ax_cam.imshow(Image.fromarray(heatmap).resize((256, 256)))
    ax_cam.set_title('GradCAM', fontsize=9)
    ax_cam.axis('off')


benign_samples = test_df[test_df['target'] == 0].sample(3, random_state=1)
malignant_samples = test_df[test_df['target'] == 1].sample(3, random_state=1)

fig, axes = plt.subplots(4, 6, figsize=(18, 13))

for col, (_, row) in enumerate(benign_samples.iterrows()):
    visualise_row(row, axes[0, col*2], axes[0, col*2+1], 'Benign')

for col, (_, row) in enumerate(malignant_samples.iterrows()):
    visualise_row(row, axes[2, col*2], axes[2, col*2+1], 'Malignant')

for ax in axes[1].flat:
    ax.axis('off')
for ax in axes[3].flat:
    ax.axis('off')

axes[0, 0].set_ylabel('Benign samples', fontsize=11, labelpad=10)
axes[2, 0].set_ylabel('Malignant samples', fontsize=11, labelpad=10)

plt.suptitle('Original Images vs GradCAM Explanations', fontsize=13)
plt.tight_layout()
plt.show()